# Redes neurais no Keras: rede densa e rede convolucional

## Objetivo

Este notebook apresenta, passo a passo, como construir, treinar e avaliar duas redes neurais no Keras, uma rede densa e uma rede convolucional, usando o conjunto de imagens MNIST. A teoria é apresentada apenas quando necessária; o foco é o uso da biblioteca.

## Como utilizar

- Execute as células na ordem em que aparecem.
- Execute cada célula somente depois que o professor indicar o passo correspondente.
- Cada célula de código é precedida de uma explicação do que ela faz e seguida de uma interpretação do resultado esperado.
- O treino da rede convolucional pode ser mais lento. Se necessário, o Colab permite ativar uma GPU no menu do ambiente de execução (opcional).

## A receita do Keras

Todo modelo no Keras segue os mesmos sete passos: **1** carregar os dados, **2** preparar os dados, **3** montar o modelo, **4** compilar, **5** treinar, **6** avaliar e **7** prever. Nas duas redes de hoje, apenas o passo 3 muda.

## Termos básicos

- **Keras**: biblioteca que funciona como uma caixa de peças prontas para montar redes neurais em Python. O **TensorFlow** é o motor que executa as contas por baixo.
- **MNIST**: conjunto de 70 mil imagens de números escritos à mão (0 a 9), cada uma com a resposta certa, chamada **rótulo**.
- **Pixel**: cada ponto de uma imagem. Em tons de cinza, vale de 0 (sem tinta) a 255 (traço mais forte).
- **Treino e teste**: o treino (60 mil imagens) é usado para a rede aprender; o teste (10 mil imagens) é guardado à parte para medir se ela aprendeu de verdade, como uma prova com questões novas.

---
# Bloco 1: rede densa

## Passo 1 — Carregar os dados

A célula abaixo importa o MNIST pelo Keras e o carrega. O arquivo não está dentro do Keras: na primeira execução ele é baixado da internet. O comando devolve quatro variáveis: `x_treino` (imagens de treino), `y_treino` (respostas de treino), `x_teste` (imagens de teste) e `y_teste` (respostas de teste).

In [ ]:
from tensorflow.keras.datasets import mnist

(x_treino, y_treino), (x_teste, y_teste) = mnist.load_data()

print(x_treino.shape, x_teste.shape)

**Interpretação.** Os formatos esperados são `(60000, 28, 28)` e `(10000, 28, 28)`: 60 mil imagens de treino e 10 mil de teste, cada uma com 28 × 28 pixels.

A célula seguinte exibe as dez primeiras imagens de treino, cada uma com a sua resposta no título.

In [ ]:
import matplotlib.pyplot as plt

fig, eixos = plt.subplots(2, 5, figsize=(10, 4.5))
for i, eixo in enumerate(eixos.ravel()):
    eixo.imshow(x_treino[i], cmap="gray_r")
    eixo.set_title(f"resposta: {y_treino[i]}")
    eixo.axis("off")
plt.tight_layout()
plt.show()

**Interpretação.** As imagens são dígitos escritos à mão por pessoas. O título de cada uma mostra o rótulo, isto é, a resposta certa.

Para ver o que o computador realmente recebe, a célula abaixo imprime os números da primeira imagem.

In [ ]:
import numpy as np

np.set_printoptions(linewidth=250)
print(x_treino[0])
print("Resposta:", y_treino[0])

**Interpretação.** Cada imagem é uma tabela de 28 × 28 números entre 0 (sem tinta) e 255 (traço mais forte). Os valores altos desenham o contorno do dígito.

## Passo 2 — Preparar os dados

Os pixels variam de 0 a 255. A rede aprende melhor com valores entre 0 e 1, por isso cada valor é dividido por 255.0. É a mesma ideia da normalização usada no KNN e nas redes do Encontro 1.

In [ ]:
x_treino_n = x_treino / 255.0
x_teste_n = x_teste / 255.0

print(x_treino_n.min(), x_treino_n.max())

**Interpretação.** A saída `0.0 1.0` confirma a nova faixa de valores. As versões normalizadas (`x_treino_n` e `x_teste_n`) são as usadas daqui em diante.

## Passo 3 — Montar o modelo (rede densa)

Uma rede densa é aquela em que cada neurônio recebe todos os valores da camada anterior. O modelo é montado como uma pilha de camadas:

| Camada | O que faz |
|---|---|
| `Sequential` | organiza as camadas em pilha, uma depois da outra |
| `Input(shape=(28, 28))` | define o formato da imagem que entra |
| `Flatten()` | estica a imagem de 28 × 28 em uma fila de 784 números |
| `Dense(128, activation="relu")` | 128 neurônios com ReLU (os mesmos neurônios do Encontro 1) |
| `Dense(10, activation="softmax")` | 10 saídas, uma por dígito; o softmax as transforma em probabilidades que somam 1 |

In [ ]:
from tensorflow import keras
from tensorflow.keras import layers

modelo_densa = keras.Sequential([
    keras.Input(shape=(28, 28)),
    layers.Flatten(),
    layers.Dense(128, activation="relu"),
    layers.Dense(10, activation="softmax"),
])

A célula seguinte exibe o resumo do modelo: as camadas, o formato de saída de cada uma e o número de parâmetros.

In [ ]:
modelo_densa.summary()

**Interpretação.** A coluna `Param #` mostra os números que cada camada aprende. A camada `Dense(128)` tem 100.480 (784 × 128 pesos mais 128 vieses) e a camada de saída tem 1.290, totalizando 101.770 parâmetros.

## Passo 4 — Compilar

A compilação define como a rede vai aprender:

- `optimizer="adam"`: o método que ajusta os pesos da rede;
- `loss="sparse_categorical_crossentropy"`: a medida do erro que a rede tenta reduzir (apropriada para várias categorias com rótulos de 0 a 9);
- `metrics=["accuracy"]`: o que se deseja acompanhar, a acurácia (porcentagem de acertos).

In [ ]:
modelo_densa.compile(optimizer="adam",
                     loss="sparse_categorical_crossentropy",
                     metrics=["accuracy"])

**Interpretação.** A célula não produz saída: apenas configura o modelo.

## Passo 5 — Treinar

Uma **época** é uma passada completa por todas as imagens de treino. Com `epochs=5`, a rede passa cinco vezes pelos dados, ajustando os pesos a cada passada. O método `fit` recebe as imagens, as respostas e o número de épocas.

In [ ]:
modelo_densa.fit(x_treino_n, y_treino, epochs=5)

**Interpretação.** A cada época o Keras exibe a perda (`loss`) e a acurácia (`accuracy`) no treino. Espera-se que a perda diminua e a acurácia aumente ao longo das épocas. Anote o tempo de cada época, exibido na saída.

## Passo 6 — Avaliar

O método `evaluate` mede o desempenho nas 10 mil imagens de teste, que a rede não viu durante o treino.

In [ ]:
perda, acuracia = modelo_densa.evaluate(x_teste_n, y_teste, verbose=0)
print(f"Acurácia no teste: {acuracia:.2%}")

**Interpretação.** A acurácia no teste é a medida que importa, pois mostra o desempenho em imagens novas. Ela pode variar um pouco de uma execução para outra.

## Passo 7 — Prever

O método `predict` devolve, para cada imagem, as 10 probabilidades (uma por dígito). A função `argmax` escolhe a posição da maior, que é o dígito previsto. A célula abaixo mostra dez imagens de teste com a previsão da rede.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

previsoes = np.argmax(modelo_densa.predict(x_teste_n, verbose=0), axis=1)

fig, eixos = plt.subplots(2, 5, figsize=(10, 4.5))
for i, eixo in enumerate(eixos.ravel()):
    eixo.imshow(x_teste[i], cmap="gray_r")
    cor = "green" if previsoes[i] == y_teste[i] else "red"
    eixo.set_title(f"previsto: {previsoes[i]} | certo: {y_teste[i]}", color=cor)
    eixo.axis("off")
plt.tight_layout()
plt.show()

**Interpretação.** Títulos verdes indicam acerto e títulos vermelhos indicam erro. O título traz o dígito previsto e o dígito certo.

## Teste com dígitos escritos à mão

Nesta etapa, a rede será testada com um dígito escrito pelo próprio aluno.

**Preparar a foto**

1. Escrever um dígito com caneta grossa, em papel claro e com boa luz.
2. Fotografar e, se necessário, recortar a foto para ficar apenas o dígito.
3. Enviar a foto para si mesmo (WhatsApp Web, e-mail ou Google Fotos) e baixá-la no computador.

**Enviar a foto ao Colab.** A célula abaixo abre uma janela para escolher o arquivo. Ela só funciona no Google Colab.

In [ ]:
import io
from PIL import Image
from google.colab import files

enviado = files.upload()          # abre a janela para escolher a foto
nome = list(enviado.keys())[0]
original = Image.open(io.BytesIO(enviado[nome]))

**Converter a foto para o formato do MNIST.** Uma foto de papel difere das imagens do MNIST: tem traço escuro sobre fundo claro, milhares de pixels, posição livre e fundo com sombras. A função abaixo converte a foto para o formato com que a rede foi treinada: tons de cinza, contraste, inversão (traço claro sobre fundo escuro), limpeza do fundo, recorte no traço, redução para caber em 20 × 20, colocação numa tela de 28 × 28 e centralização pelo centro de massa (o ponto onde está o "peso" da tinta).

In [ ]:
import numpy as np
from PIL import Image, ImageOps
from scipy import ndimage

def preparar_imagem(img):
    img = ImageOps.exif_transpose(img).convert("L")   # corrige a rotação e passa para cinza
    img = ImageOps.autocontrast(img)                  # reforça o contraste
    a = 255 - np.array(img)                           # inverte: traço claro, fundo escuro
    a = np.where(a > 100, a, 0)                       # limpa o fundo
    ys, xs = np.where(a > 0)
    a = a[ys.min():ys.max() + 1, xs.min():xs.max() + 1]   # recorta no traço
    h, w = a.shape
    k = 20 / max(h, w)                                # o dígito cabe em 20 x 20
    novo = Image.fromarray(a.astype(np.uint8)).resize((max(1, round(w * k)), max(1, round(h * k))))
    tela = Image.new("L", (28, 28), 0)                # tela 28 x 28, fundo preto
    tela.paste(novo, ((28 - novo.width) // 2, (28 - novo.height) // 2))
    a = np.array(tela)
    cy, cx = ndimage.center_of_mass(a)                # onde está o "peso" da tinta
    return ndimage.shift(a, (14 - cy, 14 - cx), order=1)   # centraliza por esse ponto

A célula seguinte aplica a função à foto enviada e mostra a imagem de 28 × 28 que a rede vai receber.

In [ ]:
import matplotlib.pyplot as plt

imagem_28 = preparar_imagem(original)

plt.imshow(imagem_28, cmap="gray_r")
plt.axis("off")
plt.show()

**Interpretação.** A imagem deve mostrar o dígito inteiro e centralizado. Se estiver cortado, apagado ou com manchas, o problema está na foto ou na conversão, e não na rede.

*Alternativa:* se não for possível enviar uma foto, a célula abaixo usa uma imagem do conjunto de teste no lugar dela. Ela só age quando nenhuma foto foi convertida; nesse caso, pule a célula anterior.

In [ ]:
if "imagem_28" not in globals():                      # só age se nenhuma foto foi convertida
    imagem_28 = x_teste[0].astype("float64")      # usa a primeira imagem do conjunto de teste

**Testar a imagem no modelo.** A foto é dividida por 255, como no treino, e recebe um formato de lote com uma imagem: `reshape(1, 28, 28)`.

In [ ]:
import numpy as np

entrada = (imagem_28 / 255.0).reshape(1, 28, 28)
probabilidades = modelo_densa.predict(entrada, verbose=0)
digito = int(np.argmax(probabilidades))

print(f"O modelo acha que é: {digito} ({probabilidades[0][digito]:.0%} de confiança)")

**Interpretação.** A saída informa o dígito previsto e a confiança, isto é, a probabilidade atribuída a esse dígito.

A célula abaixo mostra as 10 probabilidades que a rede atribuiu à foto e os três dígitos mais prováveis.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.bar(range(10), probabilidades[0])
plt.xticks(range(10))
plt.xlabel("dígito")
plt.ylabel("probabilidade")
plt.title("As 10 saídas da rede para a foto")
plt.show()

for d in np.argsort(probabilidades[0])[::-1][:3]:
    print(f"{d}: {probabilidades[0][d]:.0%}")

**Interpretação.** Mostra se o dígito certo ficou próximo (segundo ou terceiro lugar) ou distante da resposta da rede.

Para comparar a foto com dígitos do próprio MNIST, indique abaixo qual dígito foi escrito.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

digito_escrito = 6      # troque pelo dígito que você escreveu

indices = np.where(y_teste == digito_escrito)[0][:4]

fig, eixos = plt.subplots(1, 5, figsize=(12, 3))
for eixo, i in zip(eixos[:4], indices):
    eixo.imshow(x_teste[i], cmap="gray_r")
    eixo.set_title(f"{digito_escrito} do MNIST")
    eixo.axis("off")
eixos[4].imshow(imagem_28, cmap="gray_r")
eixos[4].set_title("a sua foto")
eixos[4].axis("off")
plt.tight_layout()
plt.show()

**Interpretação.** Compare a espessura do traço, o tamanho e o formato do dígito da foto com os do MNIST.

**Efeito da posição.** A célula abaixo desloca a mesma imagem em alguns pixels e mostra a resposta da rede para cada deslocamento.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import ndimage

deslocamentos = [(0, -3), (0, 0), (0, 3), (-3, 0), (3, 0)]    # (linhas, colunas), em pixels

fig, eixos = plt.subplots(1, 5, figsize=(13, 3.4))
for eixo, (dy, dx) in zip(eixos, deslocamentos):
    img = ndimage.shift(imagem_28, (dy, dx), order=1)
    probs = modelo_densa.predict((img / 255.0).reshape(1, 28, 28), verbose=0)[0]
    d = int(np.argmax(probs))
    eixo.imshow(img, cmap="gray_r")
    eixo.set_title(f"deslocado {(dy, dx)}\nprevisto: {d} ({probs[d]:.0%})")
    eixo.axis("off")
plt.tight_layout()
plt.show()

**Interpretação.** Observe se a resposta muda quando o mesmo dígito é deslocado. A rede densa trata cada pixel de forma separada, por isso a posição do dígito pode alterar o resultado.

## Resumo do Bloco 1

- A receita do Keras tem sete passos.
- Uma imagem é uma tabela de números; a rede densa a recebe esticada em uma fila de 784 valores.
- A acurácia no teste mede o desempenho em imagens novas.
- Com fotos escritas à mão, a rede pode errar, e a posição do dígito pode mudar a resposta.

---
# Bloco 2: rede convolucional

A rede densa estica a imagem em uma fila e perde a vizinhança dos pixels: um pixel e o vizinho ao lado passam a ser apenas dois números sem relação. A **rede convolucional** (CNN) olha a imagem por pedaços pequenos, preservando essa vizinhança. Ela usa três ideias:

- **Convolução**: um filtro pequeno (3 × 3) desliza pela imagem e mede quanto cada pedaço se parece com um padrão, como uma borda. Na rede, os filtros são aprendidos no treino. No Keras: `Conv2D`.
- **Pooling**: reduz a imagem, guardando o maior valor de cada bloco (por exemplo, 2 × 2). No Keras: `MaxPooling2D`.
- **Canal**: uma imagem em tons de cinza tem 1 canal; uma colorida tem 3 (vermelho, verde e azul). A CNN exige o canal explícito: o formato passa de `(28, 28)` para `(28, 28, 1)`.

## Passos 1 e 2 para a CNN

Carregar os dados e dividir por 255 é igual ao que já foi feito. Falta apenas acrescentar o canal com `reshape(-1, 28, 28, 1)`; o `-1` deixa o Python calcular a quantidade de imagens.

In [ ]:
x_treino_cnn = x_treino_n.reshape(-1, 28, 28, 1)
x_teste_cnn = x_teste_n.reshape(-1, 28, 28, 1)

print(x_treino_cnn.shape)

**Interpretação.** O formato esperado é `(60000, 28, 28, 1)`.

## Passo 3 — Montar a CNN

Comparada à rede densa, `Conv2D` e `MaxPooling2D` entram antes do `Flatten`; o resto da receita é igual.

In [ ]:
from tensorflow import keras
from tensorflow.keras import layers

modelo_cnn = keras.Sequential([
    keras.Input(shape=(28, 28, 1)),
    layers.Conv2D(32, (3, 3), activation="relu"),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(64, (3, 3), activation="relu"),
    layers.MaxPooling2D((2, 2)),
    layers.Flatten(),
    layers.Dense(10, activation="softmax"),
])

In [ ]:
modelo_cnn.summary()

**Interpretação.** O formato de saída diminui a cada camada (26 × 26, 13 × 13, 11 × 11, 5 × 5) e o `Flatten` gera 1600 valores. A primeira `Conv2D` tem 320 parâmetros, a segunda 18.496 e a camada de saída 16.010, totalizando 34.826. A CNN tem menos parâmetros que a rede densa (101.770).

## Passos 4 e 5 — Compilar e treinar a CNN

A compilação é igual à da rede densa. Anote o tempo de cada época e compare com o da rede densa.

In [ ]:
modelo_cnn.compile(optimizer="adam",
                    loss="sparse_categorical_crossentropy",
                    metrics=["accuracy"])

modelo_cnn.fit(x_treino_cnn, y_treino, epochs=5)

**Interpretação.** Observe a acurácia por época e o tempo de cada época na saída do treino.

## Passo 6 — Avaliar e comparar

In [ ]:
perda, acuracia = modelo_cnn.evaluate(x_teste_cnn, y_teste, verbose=0)
print(f"Acurácia no teste (CNN): {acuracia:.2%}")

A célula seguinte compara as duas redes quanto à acurácia no teste e ao número de parâmetros.

In [ ]:
_, acuracia_densa = modelo_densa.evaluate(x_teste_n, y_teste, verbose=0)
_, acuracia_cnn = modelo_cnn.evaluate(x_teste_cnn, y_teste, verbose=0)

print(f"Acurácia no teste | densa: {acuracia_densa:.2%} | CNN: {acuracia_cnn:.2%}")
print(f"Parâmetros        | densa: {modelo_densa.count_params()} | CNN: {modelo_cnn.count_params()}")

**Interpretação.** Compare sempre a acurácia, o tempo de treino e o número de parâmetros. Anote os valores abaixo.

| Medida | Rede densa | CNN |
|---|---|---|
| Acurácia no teste | | |
| Tempo por época | | |
| Parâmetros | 101.770 | 34.826 |

## Passo 7 — Os mesmos dígitos nas duas redes

A célula abaixo envia a mesma imagem (`imagem_28`) às duas redes. A rede densa recebe o formato `(1, 28, 28)` e a CNN, `(1, 28, 28, 1)`.

In [ ]:
import numpy as np

entrada_densa = (imagem_28 / 255.0).reshape(1, 28, 28)
entrada_cnn = (imagem_28 / 255.0).reshape(1, 28, 28, 1)

p_densa = modelo_densa.predict(entrada_densa, verbose=0)[0]
p_cnn = modelo_cnn.predict(entrada_cnn, verbose=0)[0]

print(f"Rede densa: {np.argmax(p_densa)} ({p_densa.max():.0%})")
print(f"CNN:        {np.argmax(p_cnn)} ({p_cnn.max():.0%})")

**Interpretação.** Compare as respostas e as confianças das duas redes para o mesmo dígito.

**Efeito da posição nas duas redes.** A célula seguinte repete o experimento de deslocamento, agora com as duas redes lado a lado.

In [ ]:
fig, eixos = plt.subplots(2, 5, figsize=(13, 6.4))
for col, (dy, dx) in enumerate(deslocamentos):
    img = ndimage.shift(imagem_28, (dy, dx), order=1)
    for linha, (nome, modelo, formato) in enumerate([("densa", modelo_densa, (1, 28, 28)), ("CNN", modelo_cnn, (1, 28, 28, 1))]):
        probs = modelo.predict((img / 255.0).reshape(formato), verbose=0)[0]
        d = int(np.argmax(probs))
        eixos[linha, col].imshow(img, cmap="gray_r")
        eixos[linha, col].set_title(f"{nome} | {(dy, dx)}\nprevisto: {d} ({probs[d]:.0%})")
        eixos[linha, col].axis("off")
plt.tight_layout()
plt.show()

**Interpretação.** Compare, para cada deslocamento, a resposta da rede densa (linha de cima) e da CNN (linha de baixo).

## Resumo do Bloco 2

- Os sete passos da receita são os mesmos nas duas redes.
- Só a lista de camadas muda: entram `Conv2D` e `MaxPooling2D`, e a entrada ganha um canal.
- A CNN usa filtros que deslizam pela imagem e aproveitam a vizinhança dos pixels.
- Neste exemplo, a CNN tem menos parâmetros que a rede densa (34.826 contra 101.770).

---
# Atividade opcional: repetir a receita com o Fashion-MNIST

O Fashion-MNIST tem 70 mil imagens de peças de roupa e calçados, de 28 × 28 pixels, em 10 categorias, no mesmo formato do MNIST. O carregamento usa `fashion_mnist` no lugar de `mnist` (`from tensorflow.keras.datasets import fashion_mnist`).

Realize, nas células abaixo, as etapas:

1. Carregar os dados e exibir 10 imagens com o nome da categoria. As categorias não vêm com nome; pesquise a lista na documentação do conjunto.
2. Treinar a rede densa e a rede convolucional seguindo a mesma receita.
3. Comparar acurácia e número de parâmetros.
4. Descobrir quais categorias as redes mais confundem (dica: `confusion_matrix`, do scikit-learn).
5. Escrever três ou quatro linhas comparando o resultado com o do MNIST.

**Etapa 1**

In [ ]:
# Escreva aqui o seu código

**Etapa 2**

In [ ]:
# Escreva aqui o seu código

**Etapa 3**

In [ ]:
# Escreva aqui o seu código

**Etapa 4**

In [ ]:
# Escreva aqui o seu código

**Etapa 5: comparação com o MNIST**

*(escreva aqui as suas conclusões)*

---
*Próxima etapa: a rede convolucional será aprofundada na Unidade 8.*